# Quickstart

The whole workflow on the committed data, small enough to run end to end in a
few minutes: load the published dataset, check it against the reference model,
fit a pilot model, and design peptides with it.

Nothing is downloaded and nothing is regenerated — `data/` ships with the
repository. For the published figure see `reproduce_figure.ipynb`.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using KrasPeptidePhageML, DataFrames, Printf, Random, Statistics

## 1. The data

`load_lung_dataset` reads the two CSVs in `data/` — 123 777 sequences over the
32 sequencing rounds of the lung experiment, plus the root of the selection
tree — and returns them as the `Data` object the model consumes.

In [ ]:
data, df_exp, seqs = load_lung_dataset()
@printf("%d sequences, %d nodes (root + %d rounds), %d latent\n",
        number_of_sequences(data), number_of_samples(data),
        number_of_samples(data) - 1, count(iszero, sum(data.counts; dims = 1)))
first(df_exp, 5)

## 2. The reference model, and whether we read the data correctly

`model_nn_2l.jld2` is the fitted model behind the published analysis. Its
log-likelihood on this dataset is a single number that validates the entire
CSV → `Data` chain: column alignment, the root in column 1, the all-zero latent
columns, the ancestor tuple and the one-hot encoding all have to be right for it
to come out at the published −392.4327.

In [ ]:
model_ref = load_reference_model()
ll = log_likelihood(model_ref, data)
@printf("log_likelihood = %.8f   (published: -392.4327)\n", ll)
@assert isapprox(ll, -392.4327; atol = 1e-4)

The same model reproduces the four published per-mode energy columns of the
369 designed candidates — the golden test of the vendored energy path.

In [ ]:
cand = designed_candidates()
E = panel_energies(model_ref, cand.sequence)
for (m, col) in ((:positive, :min_positive_energy), (:negative, :min_negative_energy),
                 (:mutation, :min_mutation_energy), (:wt, :min_wt_energy))
    d = maximum(abs.(E[:, MODES[m]] .- cand[!, col]))
    @printf("%-9s  max|diff| = %.2e\n", m, d)
end

## 3. Specificity labels

Every model-derived conclusion flows through absolute thresholds on
`E[mutation]` and `E[wt]`. `published_labels` applies that rule.

In [ ]:
lab = published_labels(E)
sort(combine(groupby(DataFrame(class = lab), :class), nrow => :n), :n, rev = true)

## 4. Fit a pilot model

`PILOT_SCHEDULE` is two epochs — enough to see the objective move, not enough to
reach the published optimum, which needs `DEFAULT_SCHEDULE` (400 epochs, hours).
The point here is that the training loop runs.

In [ ]:
select, washed = build_select_washed(df_exp)
model = build_model(select, washed; seed = 1)
ll0 = log_likelihood(model, data)
history, timings, _ = train_model!(model, data; schedule = PILOT_SCHEDULE)
@printf("log-likelihood  %.2f  ->  %.2f   in %.1f s\n",
        ll0, log_likelihood(model, data), sum(t.seconds for t in timings))

## 5. Design peptides

Monte-Carlo sequence design in the energy landscape: chains are annealed towards
low mutation-energy / high wt-energy and then pushed onto the Pareto front. Run
here with 40 chains per strategy; the published run used 1000.

Candidates are designed with the **reference** model — the pilot model above has
not been trained to a usable landscape.

In [ ]:
specs = default_specs(n_chains = 40, seed = 1)
df = generate_candidates(model_ref, data; specs = specs, n_top = 20, seed = 1, verbose = false)
@printf("%d unique candidates\n", nrow(df))
first(sort(df, :E_mutation), 8)

They should land where they were aimed: low `E[mutation]`, and `E[wt]` above
it for the mutant-selective strategies.

In [ ]:
@printf("median E[mutation] = %.2f,  median E[wt] - E[mutation] = %.2f\n",
        median(df.E_mutation), median(df.E_wt .- df.E_mutation))
@printf("overlap with the 369 published candidates: %d\n",
        length(intersect(df.sequence, cand.sequence)))

## Where to go next

* `scripts/01_train.jl` — the full schedule.
* `scripts/03_perturbations.jl` — the CV / bootstrap refits behind the figure.
* `reproduce_figure.ipynb` — the published robustness figure.